# Churn Revenue Optimization  
### Telecom-style churn and casino-style revenue-risk portfolio notebook

**Author:** David Johanson

## Project Goal
This notebook demonstrates how to:
- simulate customer and player behavior data
- engineer churn and value features
- estimate **revenue at risk**
- identify **high-value churners**
- evaluate a simple **retention targeting strategy**

This is a portfolio-style project built to showcase:
- business-oriented analytics
- SQL-to-Python feature logic
- retention value estimation
- clear stakeholder-friendly outputs

## Why this project matters

Many churn projects stop at model scores.  
This notebook goes one step further:

1. **Who is likely to churn?**
2. **Which churners are valuable?**
3. **How much revenue is at risk?**
4. **What value could a retention campaign save?**

That framing is much closer to how analytics is used in real business environments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Simulate telecom-style churn dataset

In a real production setting, these fields would come from:
- rolling usage aggregates
- subscriber profile tables
- campaign logs
- churn scoring outputs

Here we simulate a realistic portfolio dataset while preserving business logic.

In [ ]:
n = 5000

df = pd.DataFrame({
    "subs_id": np.arange(100000, 100000 + n),
    "tenure_days": np.random.randint(30, 3650, n),
    "avg_data_mb_30d": np.random.gamma(shape=2.5, scale=450, size=n),
    "avg_voice_min_30d": np.random.gamma(shape=2.0, scale=70, size=n),
    "avg_sms_cnt_30d": np.random.gamma(shape=1.2, scale=15, size=n),
    "total_charge_30d": np.random.gamma(shape=3.0, scale=7, size=n),
    "total_charge_90d": np.random.gamma(shape=5.0, scale=9, size=n),
    "notifications_90d": np.random.poisson(0.7, size=n),
    "days_since_last_activity": np.random.randint(0, 120, n)
})

# Build churn probability with interpretable drivers
z = (
    -2.2
    + 0.012 * df["days_since_last_activity"]
    - 0.00018 * df["avg_data_mb_30d"]
    - 0.0015 * df["avg_voice_min_30d"]
    - 0.0008 * df["tenure_days"]
    + 0.03 * df["notifications_90d"]
    - 0.01 * df["total_charge_30d"]
)

df["churn_prob"] = 1 / (1 + np.exp(-z))
df["actual_churn_flag"] = np.random.binomial(1, np.clip(df["churn_prob"], 0.01, 0.95))

df.head()

## 2. Feature engineering

These are business-friendly features frequently used in retention and campaign analytics.

In [ ]:
df["tenure_bin"] = pd.cut(
    df["tenure_days"],
    bins=[0, 365, 730, 1095, 1460, 3650],
    labels=["0-1y", "1-2y", "2-3y", "3-4y", "4y+"],
    include_lowest=True
)

df["value_segment"] = pd.cut(
    df["total_charge_90d"],
    bins=[0, 20, 50, 100, 1000],
    labels=["Low", "Medium", "High", "VIP"],
    include_lowest=True
)

df["usage_mix"] = np.where(
    df["avg_data_mb_30d"] >= df["avg_voice_min_30d"] * 10,
    "Data-heavy",
    "Balanced/Voice"
)

df["high_risk_flag"] = (df["churn_prob"] >= 0.60).astype(int)
df["high_value_flag"] = (df["total_charge_90d"] >= 20).astype(int)

feature_snapshot = df[
    [
        "subs_id", "tenure_days", "tenure_bin", "avg_data_mb_30d", "avg_voice_min_30d",
        "total_charge_90d", "value_segment", "usage_mix", "churn_prob",
        "high_risk_flag", "high_value_flag", "actual_churn_flag"
    ]
].head(10)

feature_snapshot

## 3. Portfolio KPI summary

This section answers the first business questions:
- portfolio size
- churn rate
- average 90-day revenue
- total revenue at risk from actual churners

In [ ]:
portfolio_summary = pd.DataFrame({
    "metric": [
        "Subscribers in portfolio",
        "Actual churners",
        "Actual churn rate",
        "Average 90d revenue",
        "Total 90d revenue",
        "Revenue at risk from actual churners"
    ],
    "value": [
        len(df),
        int(df["actual_churn_flag"].sum()),
        df["actual_churn_flag"].mean(),
        df["total_charge_90d"].mean(),
        df["total_charge_90d"].sum(),
        df.loc[df["actual_churn_flag"] == 1, "total_charge_90d"].sum()
    ]
})

portfolio_summary

## 4. Churn and revenue by segment

A good analyst does not only compute totals.  
They break results into actionable segments.

In [ ]:
segment_summary = (
    df.groupby(["tenure_bin", "value_segment"], observed=False)
      .agg(
          subscribers=("subs_id", "count"),
          churn_rate=("actual_churn_flag", "mean"),
          avg_90d_revenue=("total_charge_90d", "mean"),
          total_90d_revenue=("total_charge_90d", "sum")
      )
      .reset_index()
      .sort_values(["value_segment", "tenure_bin"])
)

segment_summary.head(15)

## 5. Targeting strategy

A common retention strategy is to avoid mass campaigns and focus on:
- **high churn probability**
- **revenue-active users**
- customers where intervention can produce measurable value

Target definition used here:
- `churn_prob >= 0.60`
- `total_charge_90d >= 20`

In [ ]:
target = df[
    (df["churn_prob"] >= 0.60) &
    (df["total_charge_90d"] >= 20)
].copy()

target_summary = pd.DataFrame({
    "metric": [
        "Targeted subscribers",
        "Target size %",
        "Actual churners inside target",
        "Precision within target",
        "Target 90d revenue",
        "Target revenue share %"
    ],
    "value": [
        len(target),
        len(target) / len(df),
        int(target["actual_churn_flag"].sum()),
        target["actual_churn_flag"].mean() if len(target) else 0,
        target["total_charge_90d"].sum(),
        target["total_charge_90d"].sum() / df["total_charge_90d"].sum()
    ]
})

target_summary

## 6. Value estimation (VE)

Now we estimate the business value of a retention campaign.  
Assumption: if the campaign works for some share of the target group, then the related portion of at-risk revenue is preserved.

This is not a causal model.  
It is a **scenario-based business estimate**, which is often how retention VE is first framed.

In [ ]:
actual_target_churners = target[target["actual_churn_flag"] == 1].copy()
target_revenue_at_risk = actual_target_churners["total_charge_90d"].sum()

rates = [0.10, 0.20, 0.30, 0.40]

ve = pd.DataFrame({
    "retention_success_rate": rates,
    "revenue_saved": [target_revenue_at_risk * r for r in rates]
})

ve["revenue_saved"] = ve["revenue_saved"].round(2)
ve

## 7. Revenue lift view

This view shows how much revenue can be covered by targeting the highest-risk subscribers first.

In [ ]:
lift = df.sort_values("churn_prob", ascending=False).reset_index(drop=True).copy()
lift["rank"] = np.arange(1, len(lift) + 1)
lift["population_pct"] = lift["rank"] / len(lift)
lift["cum_revenue_at_risk"] = (lift["actual_churn_flag"] * lift["total_charge_90d"]).cumsum()

total_revenue_at_risk = (df["actual_churn_flag"] * df["total_charge_90d"]).sum()
lift["cum_revenue_capture_pct"] = np.where(
    total_revenue_at_risk > 0,
    lift["cum_revenue_at_risk"] / total_revenue_at_risk,
    0
)

lift_points = (
    lift[lift["population_pct"].isin([0.01, 0.05, 0.10, 0.20, 0.30])]
    [["population_pct", "cum_revenue_capture_pct"]]
)

lift_points

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lift["population_pct"], lift["cum_revenue_capture_pct"])
plt.xlabel("Targeted population share")
plt.ylabel("Captured revenue-at-risk share")
plt.title("Revenue Lift Curve")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Churn probability distribution

This helps visualize how risk is spread across the portfolio.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["churn_prob"], bins=30)
plt.xlabel("Churn probability")
plt.ylabel("Subscribers")
plt.title("Distribution of churn probability")
plt.grid(True, alpha=0.3)
plt.show()

## 9. ARPU-style comparison: target vs non-target

This is the kind of quick comparison stakeholders usually ask for.

In [ ]:
comparison = pd.DataFrame({
    "group": ["Target", "Non-target"],
    "subscribers": [len(target), len(df) - len(target)],
    "avg_90d_revenue": [
        target["total_charge_90d"].mean() if len(target) else 0,
        df.loc[~df.index.isin(target.index), "total_charge_90d"].mean()
    ],
    "actual_churn_rate": [
        target["actual_churn_flag"].mean() if len(target) else 0,
        df.loc[~df.index.isin(target.index), "actual_churn_flag"].mean()
    ]
})

comparison

## 10. Casino-style extension

The same logic can be reused for casino / gaming analytics:
- subscriber → player
- total charge → bet or cash-in volume
- days since last activity → inactivity gap
- churn → player lapse / attrition

Below is a compact example built from the same portfolio frame.

In [ ]:
casino = pd.DataFrame({
    "player_id": df["subs_id"],
    "days_since_last_visit": df["days_since_last_activity"],
    "cash_in_90d": (df["total_charge_90d"] * np.random.uniform(2.5, 6.0, len(df))).round(2),
    "session_days_90d": np.random.poisson(8, len(df)),
    "casino_churn_prob": np.clip(
        0.15 + 0.006 * df["days_since_last_activity"] - 0.01 * np.random.poisson(8, len(df)),
        0, 0.99
    )
})

casino["actual_lapsed_flag"] = np.random.binomial(1, casino["casino_churn_prob"])

casino_summary = pd.DataFrame({
    "metric": [
        "Players",
        "Lapsed players",
        "Lapse rate",
        "Cash-in at risk"
    ],
    "value": [
        len(casino),
        int(casino["actual_lapsed_flag"].sum()),
        casino["actual_lapsed_flag"].mean(),
        casino.loc[casino["actual_lapsed_flag"] == 1, "cash_in_90d"].sum()
    ]
})

casino_summary

## 11. Business interpretation

### Telecom use case
The analysis shows how to move from raw churn scores to a practical retention action plan:
- isolate high-risk and revenue-active subscribers
- estimate at-risk revenue
- simulate what could be saved at different campaign success rates

### Casino use case
The same analytical structure can be adapted for player retention:
- identify lapsed or near-lapse players
- quantify cash-in at risk
- prioritize intervention on valuable players

### Portfolio takeaway
This project demonstrates not only data manipulation, but also:
- metric framing
- prioritization logic
- business-value storytelling

## 12. Suggested next improvements

If this were expanded into a full production project, next steps could include:
- train a proper classification model
- calibrate probabilities
- compare targeting rules vs model-based ranking
- estimate incremental uplift rather than scenario VE
- add campaign cost and net ROI logic
- replicate the feature layer in SQL / Oracle pipeline form

## Appendix: Example Oracle-style feature engineering SQL

```sql
WITH base_usage AS (
    SELECT
        subs_id,
        SUM(total_charge) AS total_charge_90d,
        AVG(data_usage_mb) AS avg_data_mb_30d,
        AVG(voice_usage_min) AS avg_voice_min_30d
    FROM usage_table
    WHERE day BETWEEN TRUNC(SYSDATE) - 90 AND TRUNC(SYSDATE)
    GROUP BY subs_id
),
subs_profile AS (
    SELECT
        subs_id,
        tenure_days,
        CASE
            WHEN tenure_days <= 365 THEN '0-1y'
            WHEN tenure_days <= 730 THEN '1-2y'
            WHEN tenure_days <= 1095 THEN '2-3y'
            WHEN tenure_days <= 1460 THEN '3-4y'
            ELSE '4y+'
        END AS tenure_bin
    FROM subs_table
)
SELECT
    b.subs_id,
    b.total_charge_90d,
    b.avg_data_mb_30d,
    b.avg_voice_min_30d,
    s.tenure_bin
FROM base_usage b
JOIN subs_profile s
    ON b.subs_id = s.subs_id;
```